In [0]:
%python
#display csv files preview as text
spark.sql("use catalog lakehouse_dev")
spark.sql("use schema datasphere_test")
spark.sql(f'''SELECT * FROM text.`/Volumes/lakehouse_dev/datasphere_test/managedvolume/babynames`''').display()

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from pyspark.sql.functions import when, col, year,current_timestamp

spark.sql("use catalog lakehouse_dev")
spark.sql("use schema datasphere_test")
csv_path = f'/Volumes/lakehouse_dev/datasphere_test/managedvolume/babynames'
babyname_csv_schema = StructType(
        [
        StructField('Year', IntegerType(), True),
        StructField('FirstName', StringType(), True),
        StructField('Country', StringType(), True),
        StructField('Sex', StringType(), True),
        StructField('Count', IntegerType(), True)
        ])

babyname_raw = (
    spark.read.format('csv').option('header', 'true').schema(babyname_csv_schema).load(csv_path).select('*', "_metadata.file_name", (year(current_timestamp())-col('Year')).alias('Age'),
                "_metadata.file_modification_time",
                current_timestamp().alias('processing_time'))
)


(babyname_raw.write
        .format("delta")
        .mode("overwrite")  #specify the save mode (overwrte, append)
        .option("overwriteSchema", "true")
        .saveAsTable("babyname_bronze")
    )
       


spark.sql(f'''SELECT * FROM babyname_bronze''').display()

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from pyspark.sql.functions import when, col, current_timestamp,year


# define table schema
def get_babyname_csv_schema():
    return StructType(
        [
        StructField('Year', StringType(), True),
        StructField('FirstName', StringType(), True),
        StructField('Country', StringType(), True),
        StructField('Sex', StringType(), True),
        StructField('Count', IntegerType(), True)
        ])
# read csv file 
def read_babyname_data(csv_path, schema):
    return (
        spark.read.format('csv').option('header', 'true').schema(schema).load(csv_path).select('*', (year(current_timestamp()) - col('Year')).alias('Age'),     "_metadata.file_name", 
                "_metadata.file_modification_time",
                current_timestamp().alias('processing_time'))
    )



# Map age column to age groups
def group_ages_map(col_name):
    return (
        when(col(col_name) < 3, "0-3")
        .when(col(col_name) < 5, "3-5")
        .when(col(col_name) < 8, "8-10")
        .when(col(col_name) >= 8, "8+")
        .otherwise("Unknown")
    )

# save dataframe to a delta table
def save_df_to_delta(dataframe, uc_table, mode):
    (dataframe.write
        .format("delta")
        .mode(mode)  #specify the save mode (overwrte, append)
        .option("overwriteSchema", "true")
        .saveAsTable(uc_table)
    )


# gold Aggregation
# create a gold-level table with aggregates counts
def get_age_agg(catalog, schema, table_name):
    query = f'''
        create or replace table {catalog}.{schema}.{table_name} as (
            select Age_Group, count(*) as Total from {catalog}.{schema}.babyname_silver group by Age_Group
        )
    '''
    return spark.sql(query)


                

In [0]:

spark.sql("use catalog lakehouse_dev")
spark.sql("use schema datasphere_test")
csv_path = f'/Volumes/lakehouse_dev/datasphere_test/managedvolume/babynames'
#read csv data into dataframe and save it to the bronze table
babyname_csv_df = read_babyname_data(csv_path, get_babyname_csv_schema())

#save the raw data as bronze table in delta format
save_df_to_delta(babyname_csv_df, 'babyname_bronze', mode='overwrite')


babyname_bronze_table = spark.table('babyname_bronze')

#transform the data by adding new columns and cleaning up metadata
silver_df = ( babyname_bronze_table.withColumn("Age_Group", group_ages_map("Age") ).drop("file_name", "file_modification_time", "processing_time"))
             
#save the transformed data as a silver table in delta format
save_df_to_delta(silver_df, 'babyname_silver', mode='overwrite')

#Gold table
get_age_agg('lakehouse_dev', 'datasphere_test', 'babyname_gold')

In [0]:
spark.table('babyname_bronze').display()
#spark.table('babyname_silver').display()
#spark.table('babyname_gold').display()')

In [0]:
from pyspark.sql.functions import col,when
def add_new_col(df,new, s_col):
    return (df.withColumn(new, 
                          when(col(s_col)==0, 'Normal')
                          .otherwise('Unknown')))
            
df = [(0,),(1,),(-1,),(None,)]

columns = ["value"]
df = spark.createDataFrame(df,columns)
df.show()

actual_df = add_new_col(df,"new_value","value")

actual_df.show()